In [1]:
import logging
import warnings
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
from plotly.subplots import make_subplots
from scipy import stats
from scipy.stats import kendalltau, pearsonr, spearmanr
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import acf, adfuller, ccf, pacf
from ydata_profiling import ProfileReport


c:\Users\BRAIN\Desktop\ML\solar-irradiance-forecasting\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\BRAIN\AppData\Local\Temp\ipykernel_13520\1899091406.py:18: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


In [2]:
bambili = pd.read_csv("../data/01_raw/Bambili.csv")
bamenda = pd.read_csv("../data/01_raw/Bamenda.csv")
bafoussam = pd.read_csv("../data/01_raw/Bafoussam.csv")
yaounde = pd.read_csv("../data/01_raw/yaounde.csv")

In [3]:
len(bambili), len(bamenda), len(bafoussam), len(yaounde)

(27214, 27214, 27214, 27214)

In [4]:
bambili.head()

,date,Temperature,humudity,irradiance,potential,wind speed
0,19500101,20.300,73.536,637.50,4.3901,0.23750
1,19500102,20.352,73.225,837.50,4.2901,0.26750
2,19500103,20.654,75.678,791.67,4.0553,0.47449
3,19500104,21.388,73.610,850.00,4.3541,0.44123
4,19500105,21.435,74.990,841.67,4.3115,0.35958


In [5]:
bamenda.head()

,Date,Temperature,Humidity,Irradiance,potential,Wind_Speed
0,19500101,19.575,74.585,854.33,4.3880,0.19861
1,19500102,19.875,74.985,833.33,4.2688,0.19761
2,19500103,20.137,78.083,779.17,3.9913,0.53274
3,19500104,21.037,74.749,841.67,4.3115,0.32011
4,19500105,21.013,76.081,825.00,4.2261,0.12056


In [6]:
bafoussam.head()

,date,Temperature,humudity,irradiance,potential,wind speed
0,19500101,22.384,77.513,805.17,4.1294,0.12452
1,19500102,22.184,76.513,804.17,4.1194,0.13452
2,19500103,22.724,78.124,779.17,3.9913,0.20164
3,19500104,23.304,77.783,829.17,4.2474,0.41398
4,19500105,23.220,78.819,816.67,4.1834,0.43339


In [7]:
yaounde.head()

,date,Temperature,humudity,irradiance,potential,wind speed
0,19500101,24.545,89.632,650.83,3.3640,1.5157
1,19500102,24.945,86.632,670.83,3.4364,1.8157
2,19500103,24.234,91.133,641.67,3.2870,2.0908
3,19500104,25.402,87.938,687.50,3.5217,1.8240
4,19500105,25.242,89.361,679.17,3.4791,2.2644


In [24]:
# standard column mapping and merging

column_mapping = {
    "Date": "date",
    "date": "date",

    "Temperature": "temperature",

    "Humidity": "humidity",
    "humudity": "humidity",

    "Irradiance": "irradiance",
    "irradiance": "irradiance",

    "potential": "potential",

    "wind speed": "wind_speed",
    "Wind_Speed": "wind_speed"
}

# Rename columns
datasets = [bambili, bamenda, bafoussam, yaounde]

for df in datasets:
    df.rename(columns=column_mapping, inplace=True)

# Add city column
bambili["city"] = "bambili"
bamenda["city"] = "bamenda"
bafoussam["city"] = "bafoussam"
yaounde["city"] = "yaounde"

# Merge datasets
merged_df = pd.concat(
    [bambili, bamenda, bafoussam, yaounde],
    ignore_index=True
)

print(merged_df.head())
print(merged_df.columns)


       date  temperature  humidity  irradiance  potential  wind_speed     city
0  19500101       20.300    73.536      637.50     4.3901     0.23750  bambili
1  19500102       20.352    73.225      837.50     4.2901     0.26750  bambili
2  19500103       20.654    75.678      791.67     4.0553     0.47449  bambili
3  19500104       21.388    73.610      850.00     4.3541     0.44123  bambili
4  19500105       21.435    74.990      841.67     4.3115     0.35958  bambili
Index(['date', 'temperature', 'humidity', 'irradiance', 'potential',
       'wind_speed', 'city'],
      dtype='object')


In [25]:
len(merged_df)

108856

In [26]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 108856 entries, 0 to 108855
Data columns (total 7 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   date         108856 non-null  int64  
 1   temperature  108856 non-null  float64
 2   humidity     108856 non-null  float64
 3   irradiance   108856 non-null  float64
 4   potential    108856 non-null  float64
 5   wind_speed   108856 non-null  float64
 6   city         108856 non-null  object 
dtypes: float64(5), int64(1), object(1)
memory usage: 5.8+ MB


In [27]:
merged_df.isna().sum()

date           0
temperature    0
humidity       0
irradiance     0
potential      0
wind_speed     0
city           0
dtype: int64

In [28]:
merged_df["date"] = pd.to_datetime(merged_df["date"], format = "%Y%m%d", errors = "coerce")
merged_df = merged_df.set_index("date")
merged_df = merged_df.sort_index()

In [29]:
merged_df.head(3)

,temperature,humidity,irradiance,potential,wind_speed,city
date,,,,,,
1950-01-01,20.300,73.536,637.50,4.3901,0.23750,bambili
1950-01-01,22.384,77.513,805.17,4.1294,0.12452,bafoussam
1950-01-01,24.545,89.632,650.83,3.3640,1.51570,yaounde


In [32]:
# Performing descriptive statistics

# Features to analyze
FEATURE_COLS = [
    "temperature",
    "humidity",
    "irradiance",
    "potential",
    "wind_speed"
]

print("DESCRIPTIVE STATISTICS PER TOWN\n")

# Loop through each city
for city in merged_df["city"].unique():

    # Filter data for the current city
    city_df = merged_df[merged_df["city"] == city]

    # Select only feature columns
    tdf = city_df[FEATURE_COLS]

    print("─" * 70)
    print(f"DESCRIPTIVE STATISTICS FOR {city.upper()}")
    print("─" * 70)

    # Generate descriptive statistics
    stats_df = tdf.describe(
        percentiles=[0.05, 0.25, 0.50, 0.75, 0.95]
    ).T

    # Add skewness and kurtosis
    stats_df["skewness"] = tdf.skew()
    stats_df["kurtosis"] = tdf.kurtosis()

    # Display results
    print(stats_df.round(3).to_string())

    print("\n")

DESCRIPTIVE STATISTICS PER TOWN

──────────────────────────────────────────────────────────────────────
DESCRIPTIVE STATISTICS FOR BAMBILI
──────────────────────────────────────────────────────────────────────
               count     mean      std     min       5%      25%      50%      75%      95%       max  skewness  kurtosis
temperature  27214.0   18.822    1.526  13.843   16.719   17.617   18.630   19.807   21.669    24.400     0.546    -0.233
humidity     27214.0   91.838    7.155  64.500   76.623   87.856   95.442   96.982   98.119    99.626    -1.231     0.369
irradiance   27214.0  643.812  166.565  83.482  370.579  525.000  637.500  779.170  904.170  1008.300    -0.133    -0.543
potential    27214.0    3.298    0.853   0.428    1.898    2.689    3.266    3.991    4.632     5.165    -0.133    -0.543
wind_speed   27214.0    0.519    0.264   0.002    0.139    0.325    0.488    0.680    1.010     1.801     0.619     0.151


────────────────────────────────────────────────────────

In [35]:
# Ydata Profiling and Automated EDA report

# Output directory
OUTPUT_DIR = Path("../data/08_reporting/notebook1")
OUTPUT_DIR.mkdir(exist_ok=True)

# Features
FEATURE_COLS = [
    "temperature",
    "humidity",
    "wind_speed",
    "irradiance",
    "potential"
]

# ─────────────────────────────────────────────────────────────
# FULL COMBINED PROFILE
# ─────────────────────────────────────────────────────────────

SAMPLE_SIZE = None   # use full dataset

profile_df = (
    merged_df.sample(SAMPLE_SIZE, random_state=42)
    if SAMPLE_SIZE
    else merged_df
)

profile = ProfileReport(
    profile_df,

    title="Solar Irradiance — Cameroon (4 Cities) — EDA Report",

    dataset={
        "description": (
            "Hourly meteorological and solar irradiance data "
            "from Yaounde, Bamenda, Bambili, and Bafoussam"
        ),
        "creator": "Solar Irradiance Forecasting Project",
    },

    variables={
        "descriptions": {
            "temperature": "Air temperature (°C)",
            "humidity": "Relative humidity (%)",
            "wind_speed": "Wind speed (m/s)",
            "irradiance": "Solar irradiance (W/m²)",
            "potential": "Solar power generation potential",
            "city": "Measurement location",
            "date": "Observation date",
        }
    },

    correlations={
        "pearson": {"calculate": True},
        "spearman": {"calculate": True},
        "kendall": {"calculate": False},
        "phi_k": {"calculate": True},
    },

    missing_diagrams={
        "bar": True,
        "matrix": True,
        "heatmap": True,
    },

    explorative=True,
    progress_bar=True,
)

# Save report
report_path = OUTPUT_DIR / "eda_report.html"

profile.to_file(report_path)

print(f" Full EDA report saved to: {report_path}")


# ─────────────────────────────────────────────────────────────
# PER-CITY MINI PROFILES
# ─────────────────────────────────────────────────────────────

for city in merged_df["city"].unique():

    city_df = merged_df[
        merged_df["city"] == city
    ][FEATURE_COLS]

    city_profile = ProfileReport(
        city_df,

        title=f"EDA Report — {city.capitalize()}",

        minimal=True,
        progress_bar=False,
    )

    city_report_path = OUTPUT_DIR / f"eda_{city}.html"

    city_profile.to_file(city_report_path)

    print(f" {city.capitalize()} profile saved to: {city_report_path}")

Summarize dataset:  69%|██████▉   | 11/16 [00:10<00:00, 14.29it/s, Calculate auto correlation]    c:\Users\BRAIN\Desktop\ML\solar-irradiance-forecasting\.venv\Lib\site-packages\ydata_profiling\model\correlations.py:87: UserWarning: There was an attempt to calculate the auto correlation, but this failed.
To hide this warning, disable the calculation
(using `df.profile_report(correlations={"auto": {"calculate": False}})`
If this is problematic for your use case, please report this as an issue:
https://github.com/ydataai/ydata-profiling/issues
(include the error message: 'cannot reindex on an axis with duplicate labels')
  warnings.warn(
Export report to file: 100%|██████████| 1/1 [00:00<00:00, 79.28it/s]


 Full EDA report saved to: ..\data\08_reporting\notebook1\eda_report.html


100%|██████████| 5/5 [00:00<00:00, 25.28it/s]


 Bambili profile saved to: ..\data\08_reporting\notebook1\eda_bambili.html


100%|██████████| 5/5 [00:00<00:00, 30.68it/s]


 Bafoussam profile saved to: ..\data\08_reporting\notebook1\eda_bafoussam.html


100%|██████████| 5/5 [00:00<00:00, 47.63it/s]


 Yaounde profile saved to: ..\data\08_reporting\notebook1\eda_yaounde.html


100%|██████████| 5/5 [00:00<00:00, 36.47it/s]


 Bamenda profile saved to: ..\data\08_reporting\notebook1\eda_bamenda.html


In [38]:
# Performing GHI(Global Horizontal Irradiance) distribution analysis for each city

fig = make_subplots(
    rows=2,
    cols=2,

    subplot_titles=[city.capitalize() for city in merged_df["city"].unique()],

    shared_xaxes=True,
    shared_yaxes=True,
)

# Subplot positions
positions = [(1, 1), (1, 2), (2, 1), (2, 2)]

# define fixed colors for cities
cities = merged_df["city"].unique()
color_map = {
    city: color for city, color in zip(
        cities,
        ["royalblue", "orange", "green", "purple"]
    )
}

for (row, col), city in zip(positions, cities):

    # Use irradiance column
    irradiance = merged_df[merged_df["city"] == city]["irradiance"].dropna()

    # Keep only daytime irradiance values
    irradiance_day = irradiance[irradiance > 10]

    # Histogram
    fig.add_trace(
        go.Histogram(
            x=irradiance_day,

            nbinsx=80,

            name=city.capitalize(),

            marker_color=color_map[city],

            opacity=0.8,

            showlegend=True,
        ),

        row=row,
        col=col,
    )

    # Mean line
    fig.add_vline(
        x=irradiance_day.mean(),

        line_dash="dash",

        line_color="red",

        annotation_text=f"μ = {irradiance_day.mean():.0f}",

        row=row,
        col=col,
    )

    # Median line
    fig.add_vline(
        x=irradiance_day.median(),

        line_dash="dot",

        line_color="blue",

        annotation_text=f"M = {irradiance_day.median():.0f}",

        row=row,
        col=col,
    )

# Layout updates
fig.update_layout(
    title_text="Solar Irradiance Distribution Across Cameroonian Towns",

    height=700,

    template="plotly_white",

    bargap=0.05,

    title_x=0.5,
)

# Axis labels
fig.update_xaxes(
    title_text="Solar Irradiance (W/m²)"
)

fig.update_yaxes(
    title_text="Frequency"
)

# Display figure
fig.show()

# Save interactive HTML
fig.write_html(
    OUTPUT_DIR / "solar_irradiance_distributions.html"
)

print("Solar irradiance distribution visualization saved successfully")

Solar irradiance distribution visualization saved successfully
